# Basic ifcfill usage without categorical encoding

This notebook shows the default `IFCTransformer` workflow. Categorical columns are filled and returned as pandas categoricals, while integer, float, and datetime columns are converted into numeric IFC variables.

## Setup

Install ifcfill with the optional notebook dependencies:

```bash
pip install "ifcfill[examples]"
```

In [1]:
import pandas as pd

from ifcfill import IFCTransformer

## Create sample data

In [2]:
df = pd.DataFrame(
    {
        "age": [25, 30, None, 40, 35],
        "salary": [50_000.50, None, 75_000.00, 90_000.25, 62_000.00],
        "city": ["London", None, "Paris", "London", "Amman"],
        "joined": pd.to_datetime(
            ["2020-01-01", "2021-06-15", None, "2023-03-10", "2022-11-01"]
        ),
        "active": ["yes", "yes", "yes", "yes", "yes"],
    }
)

df

,age,salary,city,joined,active
0,25.0,50000.50,London,2020-01-01,yes
1,30.0,NaN,NaN,2021-06-15,yes
2,NaN,75000.00,Paris,NaT,yes
3,40.0,90000.25,London,2023-03-10,yes
4,35.0,62000.00,Amman,2022-11-01,yes


## Fit and transform

`cat_encoding="none"` is the default, so categorical columns remain categorical after missing-value filling. Missing categorical values are represented by the learned missing category.

In [3]:
transformer = IFCTransformer(
    int_fill="median",
    float_fill="mean",
)

transformed = transformer.fit_transform(df)
transformed

,age,salary,city,joined
0,25,50000.5000,London,18262
1,30,69250.1875,__ifcfill_missing__,18793
2,32,75000.0000,Paris,19045
3,40,90000.2500,London,19426
4,35,62000.0000,Amman,19297


The constant `active` column is dropped. Categorical missing values become the `__ifcfill_missing__` category, and the `joined` datetime column is represented as integer days from the default anchor date.

In [4]:
transformed.dtypes

age          int64
salary     float64
city      category
joined       int64
dtype: object

## Inspect what ifcfill learned

In [5]:
transformer.column_types_

{'age': 'integer',
 'salary': 'float',
 'city': 'categorical',
 'joined': 'datetime'}

In [6]:
transformer.fill_values_

{'age': 32, 'salary': 69250.1875, 'city': '__ifcfill_missing__', 'joined': 19045}

In [7]:
transformer.dropped_constants_

{'active': ('yes', 4)}

In [8]:
transformer.missing_report_

,column,type,missing_count,missing_fraction
0,age,integer,1,0.2
1,salary,float,1,0.2
2,city,categorical,1,0.2
3,joined,datetime,1,0.2
4,active,constant,0,0.0


## Restore the original structure

In [9]:
restored = transformer.inverse_transform(
    transformed,
    restore_missing=True,
    random_state=42,
)

restored

,age,salary,city,joined,active
0,NaN,50000.5,London,18262,yes
1,30,69250.1875,NaN,18793,yes
2,32,75000.0,Paris,19045,yes
3,40,NaN,London,NaN,yes
4,35,62000.0,Amman,19297,yes
